In [33]:
import torch
import torch.nn as nn

## Noise Scheduler

In [34]:
class LinearNoiseScheduler:
  def __init__(self,num_timesteps,beta_start,beta_end):
    self.num_timesteps = num_timesteps
    self.beta_start = beta_start
    self.beta_end = beta_end

    self.betas = torch.linspace(beta_start,beta_end,num_timesteps)

    self.alphas = 1.0-self.betas
    self.alpha_cum_prod = torch.cumprod(self.alphas,dim = 0)
    self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
    self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1.0-self.alpha_cum_prod)


  def add_noise(self, original, noise, t):
      original_shape = original.shape
      batch_size = original_shape[0]
      sqrt_alpha_cum_prod = self.sqrt_alpha_cum_prod[t].reshape(batch_size)
      sqrt_one_minus_alpha_cum_prod = self.sqrt_one_minus_alpha_cum_prod[t].reshape(batch_size)

      for i in range(len(original_shape) - 1):
          sqrt_alpha_cum_prod = sqrt_alpha_cum_prod.unsqueeze(-1)
          sqrt_one_minus_alpha_cum_prod = sqrt_one_minus_alpha_cum_prod.unsqueeze(-1)

      return sqrt_alpha_cum_prod * original + sqrt_one_minus_alpha_cum_prod * noise


  def sample_prev_timestep(self,xt,noise_pred,t):
    x0 = (xt-(self.sqrt_one_minus_alpha_cum_prod[t]*noise_pred))/self.sqrt_alpha_cum_prod[t]
    x0 = torch.clamp(x0,-1,1)

    mean = xt-((self.betas[t]*noise_pred)/(self.sqrt_one_minus_alpha_cum_prod[t]))
    mean = mean/torch.sqrt(self.alphas[t])


    if t == 0:
      return mean,x0
    else:
      variance = (1-self.alpha_cum_prod[t-1])/(1-self.alpha_cum_prod[t])
      variance = variance*self.betas[t]
      sigma = variance**0.5
      z = torch.randn(xt.shape).to(xt.device)
      return mean+sigma*z,x0

## UNET

In [35]:
def get_time_embedding(time_steps,t_emb_dim):
  factor = 10000*((torch.arange(0,t_emb_dim//2,device=time_steps.device)/(t_emb_dim//2)))
  t_emb = time_steps[:,None].repeat(1,t_emb_dim//2)/factor
  t_emb = torch.cat([torch.sin(t_emb),torch.cos(t_emb)],dim=-1)
  return t_emb

In [36]:
class DownBlock(nn.Module):
  def __init__(self,in_channels,out_channels,t_emb_dim,down_sample,num_heads):
    super().__init__()
    self.down_sample = down_sample
    self.resnet_conv_first = nn.Sequential(
        nn.GroupNorm(8,in_channels),
        nn.SiLU(),
        nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=1,padding=1)
    )
    self.t_emb_layers = nn.Sequential(
        nn.SiLU(),
        nn.Linear(t_emb_dim,out_channels)
    )
    self.resnet_conv_second = nn.Sequential(
        nn.GroupNorm(8,out_channels),
        nn.SiLU(),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1)
    )
    self.attention_norm = nn.GroupNorm(8,out_channels)
    self.attention = nn.MultiheadAttention(out_channels,num_heads,batch_first=True)
    self.residual_input_conv = nn.Conv2d(in_channels,out_channels,kernel_size=1)
    self.down_sample_conv = nn.Conv2d(out_channels,out_channels,kernel_size=4,stride=2,padding=1) if self.down_sample else nn.Identity()

  def forward(self,x,t_emb):
    out = x
    resnet_input = out
    out = self.resnet_conv_first(out)
    out = out+self.t_emb_layers(t_emb)[:,:,None,None]
    out = self.resnet_conv_second(out)
    out = out+self.residual_input_conv(resnet_input)

    batch_size,channels,h,w = out.shape
    in_attn = out.reshape(batch_size,channels,h*w)
    in_attn = self.attention_norm(in_attn)
    in_attn = in_attn.transpose(1,2)
    out_attn,_ = self.attention(in_attn,in_attn,in_attn)
    out_attn = out_attn.transpose(1,2).reshape(batch_size,channels,h,w)
    out = out+out_attn

    out = self.down_sample_conv(out)
    return out

In [37]:
class MidBlock(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels, t_emb_dim, num_heads):
        super().__init__()
        self.resnet_conv_first = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, in_channels),
                nn.SiLU(),
                nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1)
            ),
            nn.Sequential(
                nn.GroupNorm(8, mid_channels),
                nn.SiLU(),
                nn.Conv2d(mid_channels, out_channels, kernel_size=3, stride=1, padding=1)
            )
        ])
        self.resnet_conv_second = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, mid_channels),
                nn.SiLU(),
                nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=1, padding=1)
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
            )
        ])
        self.t_emb_layers = nn.ModuleList([
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, mid_channels)),
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels))
        ])
        self.attention_norm = nn.GroupNorm(8, mid_channels)
        self.attention = nn.MultiheadAttention(mid_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.ModuleList([
            nn.Conv2d(in_channels, mid_channels, kernel_size=1),
            nn.Conv2d(mid_channels, out_channels, kernel_size=1)
        ])

    def forward(self, x, t_emb):
        out = x
        resnet_input = out
        out = self.resnet_conv_first[0](out)
        out = out + self.t_emb_layers[0](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[0](out)
        out = out + self.residual_input_conv[0](resnet_input)

        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn)
        in_attn = in_attn.transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn

        resnet_input = out
        out = self.resnet_conv_first[1](out)
        out = out + self.t_emb_layers[1](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[1](out)
        out = out + self.residual_input_conv[1](resnet_input)

        return out


In [38]:
class UpBlock(nn.Module):
  def __init__(self,in_channels,out_channels,t_emb_dim,up_sample,num_heads):
    super().__init__()
    self.up_sample = up_sample
    self.resnet_conv_first = nn.Sequential(
        nn.GroupNorm(8,in_channels),
        nn.SiLU(),
        nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=1,padding=1)
    )
    self.t_emb_layers = nn.Sequential(
        nn.SiLU(),
        nn.Linear(t_emb_dim,out_channels)
    )
    self.resnet_conv_second = nn.Sequential(
        nn.GroupNorm(8,out_channels),
        nn.SiLU(),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1)
    )
    self.attention_norm = nn.GroupNorm(8,out_channels)
    self.attention = nn.MultiheadAttention(out_channels,num_heads,batch_first=True)
    self.residual_input_conv = nn.Conv2d(in_channels,out_channels,kernel_size=1)
    self.up_sample_conv = nn.ConvTranspose2d(in_channels//2,in_channels//2,kernel_size=4,stride=2,padding=1) if self.up_sample else nn.Identity()

  def forward(self,x,out_down,t_emb):
    x = self.up_sample_conv(x)
    x = torch.cat([x,out_down],dim=1)

    out = x
    resnet_input = out
    out = self.resnet_conv_first(out)
    out = out+self.t_emb_layers(t_emb)[:,:,None,None]
    out = self.resnet_conv_second(out)
    out = out+self.residual_input_conv(resnet_input)

    batch_size,channels,h,w = out.shape
    in_attn = out.reshape(batch_size,channels,h*w)
    in_attn = self.attention_norm(in_attn)
    in_attn = in_attn.transpose(1,2)
    out_attn,_ = self.attention(in_attn,in_attn,in_attn)
    out_attn = out_attn.transpose(1,2).reshape(batch_size,channels,h,w)
    out = out+out_attn

    return out

In [39]:

class Unet(nn.Module):
    def __init__(self, im_channels):
        super().__init__()
        self.down_channels = [32, 64, 128, 256]
        self.mid_channels = [256, 256, 128]

        self.t_emb_dim = 128
        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim)
        )

        self.init_conv = nn.Conv2d(im_channels, self.down_channels[0], kernel_size=3, padding=1)

        self.down_blocks = nn.ModuleList()
        for i in range(len(self.down_channels) - 1):
            in_channels  = self.down_channels[i]
            out_channels = self.down_channels[i + 1]
            down_sample  = (i < len(self.down_channels) - 2)
            self.down_blocks.append(
                DownBlock(in_channels, out_channels, self.t_emb_dim, down_sample, num_heads=4)
            )

        self.mid_block = MidBlock(
            self.mid_channels[0],
            self.mid_channels[1],
            self.mid_channels[2],
            self.t_emb_dim,
            num_heads=4
        )

        self.up_blocks = nn.ModuleList()
        for i in reversed(range(len(self.down_channels) - 1)):
            in_channels  = self.mid_channels[-1] + self.down_channels[i] if i == len(self.down_channels) - 2 \
                           else self.down_channels[i + 1] + self.down_channels[i]
            out_channels = self.down_channels[i]
            up_sample    = (i > 0)
            self.up_blocks.append(
                UpBlock(in_channels, out_channels, self.t_emb_dim, up_sample, num_heads=4)
            )

        self.final_conv = nn.Sequential(
            nn.GroupNorm(8, self.down_channels[0]),
            nn.SiLU(),
            nn.Conv2d(self.down_channels[0], im_channels, kernel_size=3, padding=1)
        )

    def forward(self, x, t):
        t_emb = get_time_embedding(t, self.t_emb_dim)
        t_emb = self.t_proj(t_emb)

        out = self.init_conv(x)

        down_outputs = []
        for block in self.down_blocks:
            out = block(out, t_emb)
            down_outputs.append(out)

        out = self.mid_block(out, t_emb)

        for block in self.up_blocks:
            down_out = down_outputs.pop()
            out = block(out, down_out, t_emb)

        out = self.final_conv(out)
        return out

In [40]:
if __name__ == "__main__":
    batch_size  = 2
    im_channels = 3
    im_size     = 64
    T           = 1000

    scheduler = LinearNoiseScheduler(num_timesteps=T, beta_start=1e-4, beta_end=0.02)
    model     = Unet(im_channels=im_channels)
    model.eval()

    x = torch.randn(batch_size, im_channels, im_size, im_size)
    t = torch.randint(0, T, (batch_size,))

    noise         = torch.randn_like(x)
    noisy_x       = scheduler.add_noise(x, noise, t)

    with torch.no_grad():
        noise_pred = model(noisy_x, t)

    print(f"Input shape      : {noisy_x.shape}")
    print(f"Timesteps        : {t}")
    print(f"Output shape     : {noise_pred.shape}")
    print(f"Output min/max   : {noise_pred.min():.4f} / {noise_pred.max():.4f}")
    assert noise_pred.shape == x.shape, "Output shape mismatch!"
    print("All assertions passed.")

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 32 but got size 16 for tensor number 1 in the list.